## Data View

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pfile ='/home/ec2-user/studies/hpp_datasets/gut_microbiome/gut_microbiome.parquet'
df_gut = read_parquet(pfile)
print(df_gut.shape)
df_gut.head()

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/gut_microbiome/abundance/urs/aggregated/urs_abundance.parquet'
df_urs = read_parquet(pfile)
print(df_urs.shape)
df_urs.head()

In [ ]:
# pfile = '/home/ec2-user/studies/hpp_datasets/gut_microbiome/abundance/urs/aggregated/urs_metadata.parquet'
# df_urs_metadata = read_parquet(pfile)
# print(df_urs_metadata.shape)
# df_urs_metadata.head()

# df_urs_metadata['metaname'] = 'k_' + df_urs_metadata['kingdom'] + '|' + 'p_' + df_urs_metadata['phylum'] + '|' + 'c_' + df_urs_metadata['class'] + '|' + \
#                              'o_' + df_urs_metadata['order'] + '|' + 'f_' + df_urs_metadata['family'] + '|' + 'g_' + df_urs_metadata['genus'] + '|' + 's_' + df_urs_metadata['species']
# df_urs_metadata = df_urs_metadata[['column', 'metaname']]

# urs_name_dict = df_urs_metadata.set_index('column')['metaname'].to_dict()
# df_urs = df_urs.rename(columns=urs_name_dict)

In [ ]:
df_gut_urs = pd.merge(df_urs, 
                      df_gut[['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_timestamp', 'collection_date']], 
                      on=['participant_id', 'cohort', 'research_stage', 'array_index'])
df_gut_urs = df_gut_urs.dropna(subset=['participant_id', 'cohort', 'research_stage', 'array_index'])
print(df_gut_urs.shape)
df_gut_urs.head()

In [ ]:
# Check the number and proportion of missing values
missing_analyzer = MissingValueAnalyzer(df_gut_urs)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

## Missing Value Handling

In [ ]:
gut_species_metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone']
gut_species_metadata_col = [col for col in gut_species_metadata_col if col in df_gut_urs.columns]
df_gut_urs_metadata = df_gut_urs[gut_species_metadata_col]
df_gut_urs_features = df_gut_urs.drop(columns=gut_species_metadata_col)

## Imputation

In [ ]:
impute_value = 0.0001
min_samples = 500

# detected = (df_gut_urs_features > 0) # 0 和 nan 为 false
detected = ~df_gut_urs_features.isna() # nan 为 false，0 视为有意义值
detected_counts = detected.sum(axis=0)  # 每列（物种）被检测到的样本数
keep_species = detected_counts[detected_counts >= min_samples].index
df_gut_urs_features_filtered = df_gut_urs_features.loc[:, keep_species].copy()

df_gut_urs_features_imputed = df_gut_urs_features_filtered.fillna(impute_value)
print(df_gut_urs_features_imputed.shape)
df_gut_urs_features_imputed.head()

In [ ]:
df_gut_urs_metadata.reset_index(drop=True, inplace=True)
df_gut_urs = pd.concat([df_gut_urs_metadata, df_gut_urs_features_imputed], axis=1)
df_gut_urs.to_parquet('pheno_data/202512/013-gut_microbiome_imputed.parquet')

In [ ]:
pfile = 'pheno_data/202512/013-gut_microbiome_imputed.parquet'
df_gut = pd.read_parquet(pfile)
print(df_gut.shape)
df_gut.head()